# RAG-Based Profile Matching — Experimentation

This notebook inspects the synthetic resume corpus, demonstrates section-aware chunking and metadata extraction, builds the local ChromaDB index, then evaluates retrieval against the curated relevance labels. The embedding model is downloaded on first use. Run from the project root after installing `requirements.txt`.

In [ ]:
from pathlib import Path
import json
from resume_rag import ResumeRAG, chunk_sections, extract_metadata, parse_sections, read_resume
from job_matcher import JobMatcher
from benchmark import run_benchmark

RESUMES = Path('data/resumes')
JOBS = Path('data/jobs')
DB = Path('vector_db')
resume_paths = sorted(RESUMES.glob('*.txt'))
job_paths = sorted(JOBS.glob('*.txt'))
print(f'{len(resume_paths)} synthetic resumes; {len(job_paths)} job descriptions')

## Inspect the document pipeline

Chunking is section-aware: a chunk is never assembled from two different resume sections. Metadata extraction is intentionally heuristic and should be checked against source documents.

In [ ]:
sample_path = resume_paths[0]
sample_text = read_resume(sample_path)
sample_sections = parse_sections(sample_text)
sample_chunks = chunk_sections(sample_sections, chunk_size=60, overlap=10)
print('Metadata:', json.dumps(extract_metadata(sample_text, sample_path), indent=2))
print('First section-aware chunks:')
for chunk in sample_chunks[:3]:
    print(f"[{chunk['section']}] {chunk['text']}")

## Build embeddings and query the vector database

The next cell creates (or refreshes) each resume's chunks in a persistent ChromaDB collection. The first model load may take a few minutes and requires access to Hugging Face. Re-running ingestion updates indexed files rather than adding duplicate chunks.

In [ ]:
rag = ResumeRAG(persist_directory=DB)
ingestion = rag.ingest_directory(RESUMES)
print(ingestion)
matcher = JobMatcher(rag)
example_job = (JOBS / 'ml_engineer.txt').read_text(encoding='utf-8')
example_result = matcher.match(example_job, top_k=5)
for candidate in example_result['top_matches']:
    print(candidate['candidate_name'], candidate['match_score'], candidate['matched_skills'])
    print(' ', candidate['reasoning'])

## Evaluation and analysis

Recall@K is computed using the included synthetic relevance labels. Latency is end-to-end matching time after model load (so it excludes initial model download and ingestion). The corpus is intentionally small, and matching currently queries the complete chunk collection before hybrid reranking to avoid losing keyword-relevant candidates. Do not interpret these numbers as production or hiring-quality evidence.

In [ ]:
metrics = run_benchmark(
    jobs_directory=JOBS,
    ground_truth_path='data/evaluation_ground_truth.json',
    database_directory=DB,
    top_k=10,
)
print(json.dumps(metrics, indent=2))

### Interpretation

- **Macro recall@10** gives each job equal weight; **micro recall@10** pools all labeled relevant candidates.
- **Mean and p95 latency** depend on the local machine, embedding model, cache, and number of indexed chunks.
- The labels are curated for this toy corpus, not independently judged. Before real use, obtain consent, assess fairness and privacy risks, validate extracted fields, and evaluate with blinded human relevance judgments.
- The 0–100 score is a weighted ranking signal (semantic similarity, skill coverage, and token overlap), not a probability of job success.